# Validação causal dos parâmetros \(\theta\)

## Finalidade

A auditoria das 10.000 soluções mostrou que \(\theta_2,\theta_3,\theta_{14}\) e \(\theta_{17}\) aparecem fortemente associados à menor energia e ao bitstring ótimo.

Este teste verifica uma pergunta mais forte:

> **Quando somente um desses parâmetros é alterado, mantendo os outros 29 fixos, a energia e o bitstring realmente mudam?**

### Onde colocar

Use estas células **no final do `01_generate_dataset.ipynb`**, depois de já existirem:

```python
ansatz_base
ising
offset
exact_result
```

O arquivo `results/merged.pkl` também deve existir.

Mantenha no notebook principal:

```python
RUN_GENERATION = False
RUN_MERGE = False
```

Esta seção não chama COBYLA, não gera novamente os 10.000 vetores e não sobrescreve `merged.pkl`.


## Lógica científica

### Necessidade

Parte de um vetor ótimo e leva apenas \(\theta_j\) para uma região angular associada às soluções ruins.

- resultado esperado: \(\Delta E>0\);
- resultado esperado: \(\Delta P(x^\star)<0\).

Se isso ocorrer de forma consistente, o parâmetro ajuda a **preservar** a solução.

### Suficiência

Parte de um vetor subótimo e leva apenas \(\theta_j\) para uma região angular associada às boas soluções.

- resultado esperado: \(\Delta E<0\);
- resultado esperado: \(\Delta P(x^\star)>0\).

Se isso ocorrer, o parâmetro ajuda a **recuperar** a solução.

### Intervenção conjunta

Também são modificados simultaneamente \(\{\theta_2,\theta_3,\theta_{14},\theta_{17}\}\). Se o grupo funcionar melhor que cada parâmetro isolado, há evidência de **controle distribuído**.


In [ ]:

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from qiskit.quantum_info import Statevector

# Parâmetros identificados pela auditoria empírica.
# A indexação começa em zero: theta_02 é o terceiro valor do vetor.
CANDIDATE_THETAS = [2, 3, 14, 17]

# O código escolherá automaticamente dois parâmetros pouco associados
# como controles negativos.
N_CONTROL_THETAS = 2

# 70% localiza regiões angulares boas/ruins.
# 30% permanece separado para validar as intervenções.
DISCOVERY_FRACTION = 0.70

# Mesma discretização usada na auditoria anterior.
N_ANGLE_BINS = 24
MIN_RUNS_PER_BIN = 30

# Quantidade de vetores ótimos e subótimos confirmados por Statevector.
# Comece com 20; depois aumente para 50 para fortalecer a estatística.
N_TEST_VECTORS = 20

RANDOM_STATE = 42

CAUSAL_OUTPUT = Path('results/causal_validation')
FIGURE_OUTPUT = CAUSAL_OUTPUT / 'figures'
TABLE_OUTPUT = CAUSAL_OUTPUT / 'tables'
FIGURE_OUTPUT.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 120,
    'axes.titlesize': 15,
    'axes.labelsize': 12,
    'font.size': 11,
})

# Trava: impede que o teste seja executado enquanto a geração está ativa.
if globals().get('RUN_GENERATION', False):
    raise RuntimeError('Defina RUN_GENERATION = False antes da validação.')
if globals().get('RUN_MERGE', False):
    raise RuntimeError('Defina RUN_MERGE = False para preservar merged.pkl.')

print('Validação configurada. Nenhum COBYLA será executado.')


# 2. Verificar o experimento original e carregar as 10.000 soluções

O teste reutiliza exatamente o circuito e o Hamiltoniano que já estão na memória. O banco é apenas lido; não é recriado.


In [ ]:

# ============================================================
# 2. OBJETOS ORIGINAIS E BANCO SALVO
# ============================================================

required_objects = ['ansatz_base', 'ising', 'offset', 'exact_result']
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError('Execute primeiro as células que criam: ' + ', '.join(missing))

MERGED_PATH = Path('results/merged.pkl')
if not MERGED_PATH.is_file():
    raise FileNotFoundError('Não encontrei results/merged.pkl.')

dataset = pd.read_pickle(MERGED_PATH).copy()
required_columns = [
    'best_parameters',
    'objective_function_value',
    'most_frequent_bitstring',
]
missing_columns = [c for c in required_columns if c not in dataset.columns]
if missing_columns:
    raise KeyError('Colunas ausentes: ' + ', '.join(missing_columns))

print('Arquivo:', MERGED_PATH.resolve())
print('Linhas carregadas:', len(dataset))


# 3. Confirmar o significado dos índices

A ordem usada na intervenção é a própria ordem do circuito:

```python
PARAMETER_ORDER = list(ansatz_base.parameters)
```

Assim, `theta_vector[17]` é ligado explicitamente ao parâmetro 17 dessa ordem. A tabela também tenta mostrar em quais portas e qubits os parâmetros aparecem.


In [ ]:

# ============================================================
# 3. IDENTIDADE ESTRUTURAL
# ============================================================

PARAMETER_ORDER = list(ansatz_base.parameters)
N_THETAS = len(PARAMETER_ORDER)
if N_THETAS != 30:
    raise RuntimeError(f'O circuito possui {N_THETAS} parâmetros; eram esperados 30.')

print('ORDEM REAL DO CIRCUITO')
for index, parameter in enumerate(PARAMETER_ORDER):
    print(f'theta_{index:02d} -> {parameter.name}')

parameter_to_index = {p: i for i, p in enumerate(PARAMETER_ORDER)}
identity_rows = []

for instruction_position, instruction in enumerate(ansatz_base.data):
    operation = instruction.operation
    qubits = tuple(ansatz_base.find_bit(q).index for q in instruction.qubits)

    for expression in operation.params:
        for parameter in getattr(expression, 'parameters', set()):
            if parameter not in parameter_to_index:
                continue
            theta_index = parameter_to_index[parameter]
            identity_rows.append({
                'theta_index': theta_index,
                'theta_name': f'theta_{theta_index:02d}',
                'parameter_name': parameter.name,
                'gate': operation.name.upper(),
                'qubits': str(qubits),
                'instruction_position': instruction_position,
            })

theta_identity = pd.DataFrame(identity_rows)
theta_identity.to_csv(TABLE_OUTPUT / 'theta_identity.csv', index=False)

if theta_identity.empty:
    print('Os parâmetros não ficaram expostos nas instruções decompostas.')
    print('A ordem ainda foi confirmada por list(ansatz_base.parameters).')
else:
    display(theta_identity.loc[theta_identity['theta_index'].isin(CANDIDATE_THETAS)])


# 4. Construir a matriz \((10000,30)\)

Os ângulos são envolvidos em \([-\pi,\pi)\), pois valores separados por \(2\pi\) são equivalentes. A célula também calcula a energia e o bitstring exatos usando `exact_result`.


In [ ]:

# ============================================================
# 4. MATRIZ DOS VETORES E REFERÊNCIA EXATA
# ============================================================

def parse_theta_vector(value):
    """Converte best_parameters em um vetor NumPy simples."""
    if isinstance(value, np.ndarray):
        return value.astype(float).ravel()
    if isinstance(value, (list, tuple, pd.Series)):
        return np.asarray(value, dtype=float).ravel()
    if isinstance(value, str):
        text = value.strip()
        try:
            parsed = json.loads(text)
        except Exception:
            parsed = ast.literal_eval(text)
        return np.asarray(parsed, dtype=float).ravel()
    raise TypeError(f'Tipo inválido em best_parameters: {type(value)}')


def wrap_angle(values):
    """Coloca os ângulos no intervalo [-pi, pi)."""
    values = np.asarray(values, dtype=float)
    return (values + np.pi) % (2 * np.pi) - np.pi


theta_list = dataset['best_parameters'].map(parse_theta_vector)
if theta_list.map(len).nunique() != 1:
    raise ValueError('Os vetores theta possuem comprimentos diferentes.')

THETA = wrap_angle(np.vstack(theta_list.to_numpy()))
ENERGY_SAVED = pd.to_numeric(
    dataset['objective_function_value'], errors='coerce'
).to_numpy(dtype=float)
BITSTRING_SAVED = (
    dataset['most_frequent_bitstring']
    .astype(str)
    .str.replace(' ', '', regex=False)
    .to_numpy()
)

valid = np.all(np.isfinite(THETA), axis=1) & np.isfinite(ENERGY_SAVED)
dataset = dataset.loc[valid].reset_index(drop=True)
THETA = THETA[valid]
ENERGY_SAVED = ENERGY_SAVED[valid]
BITSTRING_SAVED = BITSTRING_SAVED[valid]

EXACT_ENERGY = float(np.real(exact_result.eigenvalue) + float(np.real(offset)))
exact_state_dict = exact_result.eigenstate.to_dict()
OPTIMAL_BITSTRINGS = sorted([
    str(bitstring)
    for bitstring, amplitude in exact_state_dict.items()
    if abs(amplitude) > 1e-12
])
EXACT_BITSTRING = OPTIMAL_BITSTRINGS[0]

if THETA.shape != (10_000, 30):
    raise RuntimeError(f'Era esperado (10000, 30), mas foi encontrado {THETA.shape}.')

dataset['energy_gap_saved'] = np.abs(ENERGY_SAVED - EXACT_ENERGY)
dataset['is_optimal_saved'] = np.isin(BITSTRING_SAVED, OPTIMAL_BITSTRINGS)

print('Formato confirmado:', THETA.shape)
print('Energia exata:', EXACT_ENERGY)
print('Bitstring(s) ótimo(s):', OPTIMAL_BITSTRINGS)
print('Taxa ótima salva:', f"{100 * dataset['is_optimal_saved'].mean():.2f}%")


# 5. Separar descoberta e validação

O split evita descobrir e validar a mesma faixa angular com os mesmos vetores.


In [ ]:

# ============================================================
# 5. SPLIT 70/30
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)
positions = rng.permutation(len(dataset))
cut = int(DISCOVERY_FRACTION * len(dataset))

discovery_positions = positions[:cut]
validation_positions = positions[cut:]

discovery_df = dataset.iloc[discovery_positions].reset_index(drop=True)
validation_df = dataset.iloc[validation_positions].reset_index(drop=True)
THETA_DISCOVERY = THETA[discovery_positions]
THETA_VALIDATION = THETA[validation_positions]

print('Descoberta:', len(discovery_df))
print('Validação:', len(validation_df))


# 6. Descobrir regiões boas, ruins e controles

Para cada \(\theta_j\), o intervalo angular é dividido em 24 faixas. Em cada faixa são calculados o gap mediano e a taxa do bitstring ótimo.

- **boa:** menor gap e maior taxa ótima;
- **ruim:** maior gap e menor taxa ótima;
- **controle:** parâmetro com baixo contraste empírico e cobertura angular suficiente.


In [ ]:

# ============================================================
# 6. REGIÕES ANGULARES
# ============================================================

angle_edges = np.linspace(-np.pi, np.pi, N_ANGLE_BINS + 1)
angle_centers = 0.5 * (angle_edges[:-1] + angle_edges[1:])
region_rows = []

for theta_index in range(N_THETAS):
    values = THETA_DISCOVERY[:, theta_index]
    bin_ids = np.digitize(values, angle_edges, right=False) - 1
    bin_ids = np.clip(bin_ids, 0, N_ANGLE_BINS - 1)

    for bin_id in range(N_ANGLE_BINS):
        local_positions = np.flatnonzero(bin_ids == bin_id)
        if len(local_positions) < MIN_RUNS_PER_BIN:
            continue

        local_df = discovery_df.iloc[local_positions]
        region_rows.append({
            'theta_index': theta_index,
            'theta_name': f'theta_{theta_index:02d}',
            'bin_id': bin_id,
            'angle_center': float(angle_centers[bin_id]),
            'angle_left': float(angle_edges[bin_id]),
            'angle_right': float(angle_edges[bin_id + 1]),
            'n_runs': int(len(local_positions)),
            'median_gap': float(local_df['energy_gap_saved'].median()),
            'optimal_rate': float(local_df['is_optimal_saved'].mean()),
        })

region_table = pd.DataFrame(region_rows)
choice_rows = []

for theta_index, group in region_table.groupby('theta_index', sort=True):
    if len(group) < 2:
        continue

    good = group.sort_values(
        ['median_gap', 'optimal_rate'], ascending=[True, False]
    ).iloc[0]
    bad = group.sort_values(
        ['median_gap', 'optimal_rate'], ascending=[False, True]
    ).iloc[0]

    choice_rows.append({
        'theta_index': int(theta_index),
        'theta_name': f'theta_{int(theta_index):02d}',
        'valid_regions': int(len(group)),
        'good_angle': float(good['angle_center']),
        'good_left': float(good['angle_left']),
        'good_right': float(good['angle_right']),
        'good_median_gap': float(good['median_gap']),
        'good_optimal_rate': float(good['optimal_rate']),
        'good_n': int(good['n_runs']),
        'bad_angle': float(bad['angle_center']),
        'bad_left': float(bad['angle_left']),
        'bad_right': float(bad['angle_right']),
        'bad_median_gap': float(bad['median_gap']),
        'bad_optimal_rate': float(bad['optimal_rate']),
        'bad_n': int(bad['n_runs']),
        'energy_contrast': float(bad['median_gap'] - good['median_gap']),
        'bitstring_contrast': float(good['optimal_rate'] - bad['optimal_rate']),
    })

region_choices = pd.DataFrame(choice_rows)
for theta_index in CANDIDATE_THETAS:
    if theta_index not in set(region_choices['theta_index']):
        raise RuntimeError(f'theta_{theta_index:02d} não possui regiões suficientes.')

energy_scale = max(region_choices['energy_contrast'].max(), 1e-12)
bitstring_scale = max(region_choices['bitstring_contrast'].max(), 1e-12)
region_choices['combined_contrast'] = (
    0.5 * region_choices['energy_contrast'] / energy_scale
    + 0.5 * region_choices['bitstring_contrast'] / bitstring_scale
)

control_table = (
    region_choices.loc[
        ~region_choices['theta_index'].isin(CANDIDATE_THETAS)
        & (region_choices['valid_regions'] >= 3)
    ]
    .sort_values('combined_contrast', ascending=True)
    .head(N_CONTROL_THETAS)
)
CONTROL_THETAS = control_table['theta_index'].astype(int).tolist()
TEST_THETAS = CANDIDATE_THETAS + CONTROL_THETAS

region_table.to_csv(TABLE_OUTPUT / 'all_angular_regions.csv', index=False)
region_choices.to_csv(TABLE_OUTPUT / 'good_bad_regions.csv', index=False)

print('Candidatos:', CANDIDATE_THETAS)
print('Controles:', CONTROL_THETAS)
display(region_choices.loc[region_choices['theta_index'].isin(TEST_THETAS)])


# 7. Avaliador exato

A função liga explicitamente `theta_vector[j]` ao parâmetro `PARAMETER_ORDER[j]`, constrói o `Statevector` e mede energia, probabilidade ótima e bitstring dominante. Não há ruído de shots nesta etapa.


In [ ]:

# ============================================================
# 7. AVALIAÇÃO EXATA SEM COBYLA E SEM SHOTS
# ============================================================

def evaluate_theta_exact(theta_vector):
    theta_vector = wrap_angle(np.asarray(theta_vector, dtype=float))
    if len(theta_vector) != N_THETAS:
        raise ValueError(f'O vetor possui {len(theta_vector)} valores.')

    # Ligação explícita: o índice do banco é o índice usado no circuito.
    parameter_map = {
        parameter: theta_vector[index]
        for index, parameter in enumerate(PARAMETER_ORDER)
    }
    bound_circuit = ansatz_base.assign_parameters(parameter_map, inplace=False)
    state = Statevector.from_instruction(bound_circuit)

    energy = float(
        np.real(state.expectation_value(ising)) + float(np.real(offset))
    )
    probabilities = state.probabilities_dict()
    p_optimal = float(sum(
        probabilities.get(bitstring, 0.0)
        for bitstring in OPTIMAL_BITSTRINGS
    ))
    dominant_bitstring = max(probabilities, key=probabilities.get)

    return {
        'energy': energy,
        'energy_gap': abs(energy - EXACT_ENERGY),
        'p_optimal': p_optimal,
        'dominant_bitstring': dominant_bitstring,
        'dominant_probability': float(probabilities[dominant_bitstring]),
        'dominant_is_optimal': dominant_bitstring in OPTIMAL_BITSTRINGS,
    }

quick_test = evaluate_theta_exact(THETA_VALIDATION[0])
quick_test


# 8. Confirmar vetores ótimos e subótimos

A classificação salva veio de 4.096 shots. Antes da intervenção, cada caso é reavaliado pelo `Statevector`. Só entram no teste vetores cuja família é confirmada exatamente.


In [ ]:

# ============================================================
# 8. CASOS CONFIRMADOS
# ============================================================

saved_optimal_mask = validation_df['is_optimal_saved'].to_numpy(dtype=bool)
optimal_positions = np.flatnonzero(saved_optimal_mask)
suboptimal_positions = np.flatnonzero(~saved_optimal_mask)


def collect_confirmed_cases(candidate_positions, want_optimal, n_cases):
    selected = []
    for position in rng.permutation(candidate_positions):
        theta_vector = THETA_VALIDATION[position].copy()
        metrics = evaluate_theta_exact(theta_vector)
        correct_family = (
            metrics['dominant_is_optimal']
            if want_optimal
            else not metrics['dominant_is_optimal']
        )
        if not correct_family:
            continue
        selected.append({
            'position': int(position),
            'theta': theta_vector,
            'base': metrics,
        })
        if len(selected) >= n_cases:
            break
    return selected

optimal_cases = collect_confirmed_cases(
    optimal_positions, want_optimal=True, n_cases=N_TEST_VECTORS
)
suboptimal_cases = collect_confirmed_cases(
    suboptimal_positions, want_optimal=False, n_cases=N_TEST_VECTORS
)

if len(optimal_cases) < N_TEST_VECTORS:
    raise RuntimeError(f'Vetores ótimos confirmados: {len(optimal_cases)}.')
if len(suboptimal_cases) < N_TEST_VECTORS:
    raise RuntimeError(f'Vetores subótimos confirmados: {len(suboptimal_cases)}.')

print('Ótimos confirmados:', len(optimal_cases))
print('Subótimos confirmados:', len(suboptimal_cases))


# 9. Intervenções individuais e conjunta

A função `intervene_single` copia o vetor e modifica somente um índice. Portanto, os outros 29 parâmetros permanecem rigorosamente fixos.


In [ ]:

# ============================================================
# 9. EXECUÇÃO DAS INTERVENÇÕES
# ============================================================

region_lookup = region_choices.set_index('theta_index')


def intervene_single(theta_vector, theta_index, new_angle):
    """Altera somente um theta; os outros 29 permanecem fixos."""
    modified = theta_vector.copy()
    modified[theta_index] = new_angle
    return wrap_angle(modified)


def intervene_group(theta_vector, theta_indices, region_name):
    """Altera simultaneamente o grupo para as regiões good ou bad."""
    if region_name not in {'good', 'bad'}:
        raise ValueError("region_name deve ser 'good' ou 'bad'.")
    modified = theta_vector.copy()
    column = 'good_angle' if region_name == 'good' else 'bad_angle'
    for theta_index in theta_indices:
        modified[theta_index] = float(region_lookup.loc[theta_index, column])
    return wrap_angle(modified)


intervention_rows = []

for theta_index in TEST_THETAS:
    good_angle = float(region_lookup.loc[theta_index, 'good_angle'])
    bad_angle = float(region_lookup.loc[theta_index, 'bad_angle'])
    theta_type = 'candidato' if theta_index in CANDIDATE_THETAS else 'controle'

    # Necessidade: ótimo -> região ruim.
    for case_id, case in enumerate(optimal_cases):
        new_metrics = evaluate_theta_exact(
            intervene_single(case['theta'], theta_index, bad_angle)
        )
        base = case['base']
        intervention_rows.append({
            'test': 'necessidade',
            'theta_index': theta_index,
            'theta_label': f'theta_{theta_index:02d}',
            'theta_type': theta_type,
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

    # Suficiência: subótimo -> região boa.
    for case_id, case in enumerate(suboptimal_cases):
        new_metrics = evaluate_theta_exact(
            intervene_single(case['theta'], theta_index, good_angle)
        )
        base = case['base']
        intervention_rows.append({
            'test': 'suficiencia',
            'theta_index': theta_index,
            'theta_label': f'theta_{theta_index:02d}',
            'theta_type': theta_type,
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

GROUP_LABEL = 'grupo_02_03_14_17'

for test_name, cases, region_name in [
    ('necessidade', optimal_cases, 'bad'),
    ('suficiencia', suboptimal_cases, 'good'),
]:
    for case_id, case in enumerate(cases):
        new_metrics = evaluate_theta_exact(
            intervene_group(case['theta'], CANDIDATE_THETAS, region_name)
        )
        base = case['base']
        intervention_rows.append({
            'test': test_name,
            'theta_index': -1,
            'theta_label': GROUP_LABEL,
            'theta_type': 'grupo',
            'case_id': case_id,
            'base_energy': base['energy'],
            'new_energy': new_metrics['energy'],
            'delta_energy': new_metrics['energy'] - base['energy'],
            'base_p_optimal': base['p_optimal'],
            'new_p_optimal': new_metrics['p_optimal'],
            'delta_p_optimal': new_metrics['p_optimal'] - base['p_optimal'],
            'base_is_optimal': base['dominant_is_optimal'],
            'new_is_optimal': new_metrics['dominant_is_optimal'],
            'base_bitstring': base['dominant_bitstring'],
            'new_bitstring': new_metrics['dominant_bitstring'],
        })

intervention_results = pd.DataFrame(intervention_rows)
intervention_results.to_csv(TABLE_OUTPUT / 'intervention_results.csv', index=False)
print('Intervenções concluídas:', len(intervention_results))
display(intervention_results.head())


# 10. Resumo estatístico

O teste de Wilcoxon é pareado: compara cada vetor antes e depois da intervenção. A taxa de direção mostra em quantos casos a mudança ocorreu no sentido esperado.


In [ ]:

# ============================================================
# 10. RESUMO E TESTE PAREADO
# ============================================================

def safe_wilcoxon(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan
    if np.allclose(values, 0.0):
        return 1.0
    try:
        return float(wilcoxon(values).pvalue)
    except Exception:
        return np.nan

summary_rows = []

for (test_name, theta_label, theta_type), group in intervention_results.groupby(
    ['test', 'theta_label', 'theta_type'], sort=False
):
    delta_energy = group['delta_energy'].to_numpy(dtype=float)
    delta_probability = group['delta_p_optimal'].to_numpy(dtype=float)

    if test_name == 'necessidade':
        expected_energy_rate = float(np.mean(delta_energy > 0))
        expected_probability_rate = float(np.mean(delta_probability < 0))
        bitstring_success_rate = float(np.mean(~group['new_is_optimal'].to_numpy(bool)))
    else:
        expected_energy_rate = float(np.mean(delta_energy < 0))
        expected_probability_rate = float(np.mean(delta_probability > 0))
        bitstring_success_rate = float(np.mean(group['new_is_optimal'].to_numpy(bool)))

    if expected_energy_rate >= 0.70 and expected_probability_rate >= 0.70:
        conclusion = 'efeito consistente'
    elif expected_energy_rate >= 0.60 or expected_probability_rate >= 0.60:
        conclusion = 'efeito parcial'
    else:
        conclusion = 'associação não confirmada causalmente'

    summary_rows.append({
        'test': test_name,
        'theta_label': theta_label,
        'theta_type': theta_type,
        'n_vectors': int(len(group)),
        'median_delta_energy': float(np.median(delta_energy)),
        'median_delta_p_optimal': float(np.median(delta_probability)),
        'expected_energy_rate': expected_energy_rate,
        'expected_probability_rate': expected_probability_rate,
        'bitstring_success_rate': bitstring_success_rate,
        'p_value_energy': safe_wilcoxon(delta_energy),
        'p_value_probability': safe_wilcoxon(delta_probability),
        'automatic_conclusion': conclusion,
    })

causal_summary = pd.DataFrame(summary_rows)
causal_summary.to_csv(TABLE_OUTPUT / 'causal_summary.csv', index=False)
display(causal_summary.sort_values(['test', 'theta_type', 'theta_label']))


# 11. Gráficos autocomentados

- necessidade: resultado esperado \(\Delta E>0\) e \(\Delta P^\star<0\);
- suficiência: resultado esperado \(\Delta E<0\) e \(\Delta P^\star>0\).


In [ ]:

# ============================================================
# 11. FUNÇÃO ÚNICA PARA OS QUATRO GRÁFICOS
# ============================================================

def plot_causal_summary(test_name, metric, filename):
    table = causal_summary.loc[causal_summary['test'] == test_name].copy()

    if metric == 'energy':
        column = 'median_delta_energy'
        factor = 1.0
        xlabel = 'ΔE mediano = valor modificado − valor original'
        expected = 'ΔE > 0' if test_name == 'necessidade' else 'ΔE < 0'
    else:
        column = 'median_delta_p_optimal'
        factor = 100.0
        xlabel = 'ΔP(bitstring ótimo), em pontos percentuais'
        expected = 'ΔP* < 0' if test_name == 'necessidade' else 'ΔP* > 0'

    table['plot_value'] = factor * table[column]
    table = table.sort_values('plot_value', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(table['theta_label'], table['plot_value'])
    ax.axvline(0.0, linestyle='--', linewidth=1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Intervenção')
    ax.set_title(f'{test_name.capitalize()} — {metric}\nResultado esperado: {expected}')
    ax.grid(axis='x', alpha=0.25)

    for bar, value in zip(bars, table['plot_value']):
        suffix = ' p.p.' if metric == 'probability' else ''
        ax.text(value, bar.get_y() + bar.get_height()/2, f' {value:.4f}{suffix}', va='center')

    # Escolhe o maior efeito na direção esperada.
    if test_name == 'necessidade' and metric == 'energy':
        strongest = table.sort_values('plot_value', ascending=False).iloc[0]
    elif test_name == 'necessidade' and metric == 'probability':
        strongest = table.sort_values('plot_value', ascending=True).iloc[0]
    elif test_name == 'suficiencia' and metric == 'energy':
        strongest = table.sort_values('plot_value', ascending=True).iloc[0]
    else:
        strongest = table.sort_values('plot_value', ascending=False).iloc[0]

    ax.text(
        0.98, 0.05,
        'LEITURA AUTOMÁTICA\n'
        f"Maior efeito: {strongest['theta_label']}\n"
        f"Valor: {strongest['plot_value']:.4f}\n"
        f"Conclusão: {strongest['automatic_conclusion']}",
        transform=ax.transAxes,
        ha='right', va='bottom',
        bbox={'boxstyle': 'round,pad=0.5', 'facecolor': 'white', 'alpha': 0.92},
    )

    fig.tight_layout()
    fig.savefig(FIGURE_OUTPUT / filename, dpi=300, bbox_inches='tight')
    plt.show()


plot_causal_summary('necessidade', 'energy', '01_necessity_energy.png')
plot_causal_summary('necessidade', 'probability', '02_necessity_probability.png')
plot_causal_summary('suficiencia', 'energy', '03_sufficiency_energy.png')
plot_causal_summary('suficiencia', 'probability', '04_sufficiency_probability.png')


# 12. Relatório final

A conclusão mais forte ocorre quando:

1. candidatos superam os controles;
2. necessidade e suficiência apontam na direção esperada;
3. a intervenção conjunta supera as individuais.


In [ ]:

# ============================================================
# 12. CONCLUSÃO AUTOMÁTICA
# ============================================================

print('=' * 82)
print('VALIDAÇÃO CAUSAL DOS THETA')
print('=' * 82)

for theta_index in CANDIDATE_THETAS:
    label = f'theta_{theta_index:02d}'
    print('\n' + label)
    print('-' * len(label))

    for test_name in ['necessidade', 'suficiencia']:
        row = causal_summary.loc[
            (causal_summary['theta_label'] == label)
            & (causal_summary['test'] == test_name)
        ]
        if row.empty:
            continue
        row = row.iloc[0]
        print(f"{test_name.capitalize()}: {row['automatic_conclusion']}")
        print(f"  ΔE mediano: {row['median_delta_energy']:.6f}")
        print(f"  ΔP ótimo: {100 * row['median_delta_p_optimal']:.1f} p.p.")
        print(f"  Direção energética esperada: {100 * row['expected_energy_rate']:.1f}%")
        print(f"  Mudança correta do dominante: {100 * row['bitstring_success_rate']:.1f}%")

print('\nINTERVENÇÃO CONJUNTA')
display(causal_summary.loc[causal_summary['theta_label'] == GROUP_LABEL])

control_labels = [f'theta_{index:02d}' for index in CONTROL_THETAS]
print('\nCONTROLES NEGATIVOS')
display(causal_summary.loc[causal_summary['theta_label'].isin(control_labels)])

print('\nTabelas:', TABLE_OUTPUT.resolve())
print('Figuras:', FIGURE_OUTPUT.resolve())
